In [6]:
# Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [7]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [8]:
# Helpers to write history
def add_user_message(messages, text):
	user_message = {"role": "user", "content": text}
	messages.append(user_message)


def add_assistant_message(messages, text):
	assistant_message = {"role": "assistant", "content": text}
	messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
	params = {
			"model": model,
			"max_tokens": 600,
			"messages": messages,
			"temperature": temperature,
			"stop_sequences": stop_sequences,
	}
	if system:
			params["system"] = system

	message = client.messages.create(**params)
	return message.content[0].text

In [9]:
import json

def generate_dataset():
	prompt = """
	Generate an evaluation dataset for a prompt evaluation. 
	The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks.
	Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

	Example output:
		```json
				[
				{
						"task": "Description of task",
				},
			...additional
			]
		```

	* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
	* Focus on tasks that do not require writing much code

	Please generate 3 objects.
"""
	messages = []
	add_user_message(messages, prompt)
	add_assistant_message(messages, "```json")
	text = chat(messages, stop_sequences=["```"])
	return json.loads(text)

In [10]:
dataset = generate_dataset()
print(dataset)  

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-west-2/path' should return 'us-west-2'). Use regex to extract the region code."}, {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to all objects in an S3 bucket named 'my-data-bucket'."}, {'task': "Write a Python function that validates if a given string is a valid AWS EC2 instance ID format (e.g., 'i-0abcd1234efgh5678'). Return True or False."}]


In [12]:
import json

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

print('Saved to dataset.json ✓')


Saved to dataset.json ✓


In [ ]:
def run_promt(test_case):
    """Merges the promt and test case input, then retruns the result"""